# Créer et gérer son environnement de développement avec `uv`

> **Objectif** : savoir initialiser un projet Python avec `uv`, comprendre les différents modes d'initialisation, gérer ses dépendances, et comprendre les fichiers générés.

## 1. `uv init` — Initialiser un projet

La commande `uv init` crée un nouveau projet Python avec tout ce qu'il faut pour démarrer.

```bash
uv init mon-projet
cd mon-projet
```

Par défaut (sans option), `uv init` crée un **projet applicatif simple** : un script Python à exécuter, pas une bibliothèque à importer. C'est le mode le plus courant quand on démarre.

**Fichiers créés :**

```
mon-projet/
├── .python-version   ← la version de Python utilisée
├── main.py           ← point d'entrée du programme
├── pyproject.toml    ← carte d'identité du projet + dépendances
├── README.md         ← documentation (vide)
```

> **Quand utiliser `uv init` tout court ?**
> Pour un script, une analyse ponctuelle, un notebook, un petit outil — tout projet où on n'a pas besoin d'importer son propre code comme une bibliothèque.

## 2. Les variantes de `uv init` : `--app`, `--lib`, `--package`

`uv init` accepte des options qui changent la **structure** du projet généré. Chaque variante correspond à un cas d'usage différent.

### 2.1 `uv init --app` (Application)

```bash
uv init --app mon-api
```

C'est le comportement **par défaut** de `uv init` (avec ou sans `--app`, le résultat est le même). Le projet est une **application** : un programme qu'on exécute, pas un paquet qu'on importe.

```
mon-api/
├── .python-version
├── main.py             ← on lance : uv run main.py
├── pyproject.toml
├── README.md
```

**Cas d'usage** : un script d'analyse de données, une API Flask/FastAPI, un outil en ligne de commande, un projet de notebooks.

### 2.2 `uv init --lib` (Bibliothèque)

```bash
uv init --lib mon-utils
```

Crée un projet structuré comme une **bibliothèque Python** : un paquet qu'on pourra importer dans d'autres projets.

```
mon-utils/
├── .python-version
├── pyproject.toml
├── README.md
└── src/
    └── mon_utils/        ← un vrai paquet Python (avec __init__.py)
        ├── __init__.py
        └── py.typed
```

Remarquez la structure `src/mon_utils/` : le code est dans un **sous-dossier `src/`**, ce qui est la convention recommandée pour les bibliothèques (appelée *src layout*). Le fichier `__init__.py` fait de `mon_utils` un paquet importable.

**Cas d'usage** : vous développez une bibliothèque de fonctions utilitaires que plusieurs projets vont réutiliser, ou que vous voulez partager avec des collègues / publier sur PyPI.

### 2.3 `uv init --package` (Application structurée comme un paquet)

```bash
uv init --package mon-pipeline
```

C'est un **hybride** : le projet est une application (un programme qu'on exécute), mais organisé comme un paquet Python avec la structure `src/`.

```
mon-pipeline/
├── .python-version
├── pyproject.toml
├── README.md
└── src/
    └── mon_pipeline/
        ├── __init__.py
        └── main.py       ← point d'entrée, mais dans un paquet
```

Le `pyproject.toml` généré contient aussi une section `[project.scripts]` qui permet de lancer le programme directement par son nom :

```toml
[project.scripts]
mon-pipeline = "mon_pipeline.main:main"
```

```bash
uv run mon-pipeline    # lance la fonction main() de mon_pipeline/main.py
```

**Cas d'usage** : un projet de long terme qui grossit — un pipeline de données, une application métier. La structure en paquet permet d'organiser le code en modules et de faire des imports internes propres.

### 2.4 Pourquoi structurer en paquet ? L'avantage des imports internes

Avec `--package` ou `--lib`, votre code est un **paquet Python installé** dans l'environnement virtuel. Cela veut dire que vous pouvez faire des imports propres entre vos fichiers, peu importe d'où vous lancez votre script :

```
src/
└── mon_pipeline/
    ├── __init__.py
    ├── main.py
    ├── utils.py          ← fonctions utilitaires
    └── nettoyage.py      ← logique de nettoyage
```

```python
# Dans main.py, on importe depuis son propre paquet :
from mon_pipeline.utils import charger_csv
from mon_pipeline.nettoyage import nettoyer_colonnes

df = charger_csv("data/ventes.csv")
df = nettoyer_colonnes(df)
```

Sans structure de paquet (avec un simple `uv init`), ces imports ne fonctionnent pas de manière fiable — on se retrouve avec des `sys.path.append()` fragiles ou des erreurs `ModuleNotFoundError`.

> **En résumé** : pour un projet court (analyse, notebook), `uv init` suffit largement. Si le projet va durer et grossir, `uv init --package` vous évitera de restructurer plus tard.

### 2.5 Tableau récapitulatif

| Option | Structure | Installable / Importable | Cas d'usage typique |
|--------|-----------|-------------------------|---------------------|
| `uv init` (ou `--app`) | `main.py` à la racine | Non | Script, notebook, analyse ponctuelle |
| `uv init --lib` | `src/mon_paquet/` | Oui (bibliothèque) | Bibliothèque réutilisable, publication PyPI |
| `uv init --package` | `src/mon_paquet/` + `[project.scripts]` | Oui (application) | Pipeline, application métier, projet long terme |

## 3. Installer des dépendances

### 3.1 Ajouter un paquet

```bash
uv add polars
```

Cette commande :
1. Télécharge et installe `polars` (et ses dépendances) dans le `.venv/`
2. Ajoute `polars` à la liste `dependencies` dans `pyproject.toml`
3. Met à jour le fichier `uv.lock` avec les versions exactes

On peut ensuite l'utiliser dans notre code :

```python
import polars as pl

df = pl.read_csv("data/ventes.csv")
print(df.head())
```

Pour installer plusieurs paquets d'un coup :

```bash
uv add pandas matplotlib seaborn
```

Pour spécifier une contrainte de version :

```bash
uv add "polars>=1.0,<2.0"
```

### 3.2 Dépendances de développement (`--dev`)

Certaines bibliothèques ne servent que **pendant le développement** — elles ne sont pas nécessaires pour faire tourner le programme en production. Par exemple :

- **`ruff`** : linter et formateur de code (vérifie que le code est propre)
- **`pytest`** : framework de tests
- **`ipykernel`** : pour utiliser l'environnement dans Jupyter

On les installe avec le flag `--dev` :

```bash
uv add --dev ruff pytest ipykernel
```

Dans le `pyproject.toml`, elles apparaissent dans une section séparée :

```toml
[project]
dependencies = [
    "polars>=1.0",        # ← dépendances "normales" (production)
]

[dependency-groups]
dev = [
    "ruff>=0.8",          # ← dépendances de développement
    "pytest>=8.0",
    "ipykernel>=6.29",
]
```

**Pourquoi cette distinction ?**

Quand vous déployez votre application sur un serveur, vous ne voulez pas installer `ruff` ou `pytest` — ça alourdit l'installation pour rien. Avec `uv sync --no-dev`, seules les dépendances de production sont installées.

```bash
# Sur votre machine (tout installer, y compris les outils de dev) :
uv sync

# Sur un serveur de production (seulement ce qui est nécessaire) :
uv sync --no-dev
```

### 3.3 Retirer une dépendance

```bash
uv remove polars
```

Cela retire le paquet du `pyproject.toml`, du `uv.lock` et du `.venv/`.

## 4. Comprendre les fichiers générés

Après un `uv init` suivi de quelques `uv add`, votre projet contient ces fichiers :

```
mon-projet/
├── .python-version     ← version de Python (ex: "3.12")
├── .venv/              ← l'environnement virtuel (créé au premier uv run ou uv sync)
├── main.py             ← votre code
├── pyproject.toml      ← configuration du projet
├── uv.lock             ← versions exactes de toutes les dépendances
├── README.md           ← documentation
```

Voyons les plus importants.

### 4.1 `pyproject.toml` — le fichier central

C'est le fichier le plus important. Il décrit **tout** sur votre projet. Voici un exemple commenté :

```toml
[project]
name = "mon-projet"                   # Nom du projet
version = "0.1.0"                     # Version actuelle
description = ""                      # Description courte (optionnelle)
readme = "README.md"                  # Fichier de documentation
requires-python = ">=3.12"            # Version minimum de Python requise
dependencies = [                      # Les paquets dont le projet a besoin
    "pandas>=2.2",
    "polars>=1.0",
    "jupyterlab>=4.0",
]

[dependency-groups]
dev = [                               # Les paquets utilisés uniquement en dev
    "ruff>=0.8",
    "pytest>=8.0",
]

[build-system]                        # Comment construire le projet
requires = ["hatchling"]              # (géré automatiquement par uv,
build-backend = "hatchling.build"     #  vous n'y toucherez quasiment jamais)
```

**Les sections clés :**

| Section | Rôle |
|---------|------|
| `[project]` | Identité du projet : nom, version, dépendances |
| `[project].dependencies` | Ce que votre code a besoin pour fonctionner |
| `[dependency-groups]` | Dépendances optionnelles (dev, tests, docs…) |
| `[build-system]` | Comment empaqueter le projet (géré par uv) |
| `[tool.ruff]` | Configuration de ruff (si vous l'ajoutez) |

> **À retenir** : `pyproject.toml` est un fichier **texte** lisible et modifiable. Quand vous faites `uv add pandas`, uv l'édite pour vous, mais vous pouvez aussi l'éditer à la main.

### 4.2 `uv.lock` — le fichier de verrouillage

Le `pyproject.toml` dit *"je veux pandas >= 2.2"*. Mais quelle version exacte ? Et quelles versions de numpy, pytz, etc. (les dépendances de pandas) ?

C'est le rôle du **`uv.lock`** : il enregistre les **versions exactes** de chaque paquet installé, y compris toutes les dépendances transitives.

```
# Extrait simplifié d'un uv.lock
pandas==2.2.3
numpy==2.1.1
python-dateutil==2.9.0
pytz==2024.1
...
```

**En pratique :**
- `pyproject.toml` = ce que vous **voulez** (les contraintes)
- `uv.lock` = ce qui est **réellement installé** (les versions exactes)

Le `uv.lock` doit être **commité dans git** : c'est ce qui garantit que tous les développeurs et le serveur utilisent exactement les mêmes versions.

### 4.3 `.venv/` — l'environnement virtuel

Le dossier `.venv/` est créé automatiquement par `uv` (au premier `uv sync`, `uv run` ou `uv add`). Il contient :
- Une copie (ou un lien) de l'interpréteur Python
- Tous les paquets installés

Ce dossier est **local à votre machine** et ne doit **jamais** être commité dans git. Il est régénérable à tout moment avec `uv sync`.

> Le fichier `.gitignore` généré par `uv init` exclut déjà `.venv/` automatiquement.

### 4.4 `.python-version`

Un fichier texte d'une seule ligne contenant la version de Python utilisée par le projet (ex : `3.12`). `uv` le lit pour savoir quelle version de Python utiliser. Si cette version n'est pas installée sur votre machine, `uv` peut la télécharger automatiquement.

## 5. Le cache de `uv`

Quand vous faites `uv add pandas`, `uv` télécharge le paquet depuis PyPI. Mais si vous créez un deuxième projet qui utilise aussi `pandas`, il ne le re-télécharge pas : il le prend dans son **cache**.

Le cache est un dossier global sur votre machine (pas dans le projet) :
- **macOS** : `~/Library/Caches/uv/`
- **Linux** : `~/.cache/uv/`
- **Windows** : `%LOCALAPPDATA%\uv\cache\`

**Quelques commandes utiles :**

```bash
# Voir la taille du cache
uv cache dir
uv cache clean --dry-run    # montre ce qui serait supprimé

# Vider le cache (si besoin de libérer de l'espace)
uv cache clean
```

> **En pratique**, vous n'aurez quasiment jamais besoin de toucher au cache. C'est une des raisons pour lesquelles `uv` est si rapide : il évite de re-télécharger ce qui existe déjà.

## 6. Ce qu'il faut commiter dans git (et ce qu'il ne faut pas)

| Fichier | Git ? | Pourquoi |
|---------|-------|----------|
| `pyproject.toml` | **Oui** | C'est la définition du projet |
| `uv.lock` | **Oui** | Garantit la reproductibilité |
| `.python-version` | **Oui** | Pour que tout le monde utilise la même version |
| `main.py` / `src/` | **Oui** | Votre code ! |
| `.venv/` | **Non** | Régénérable avec `uv sync` — trop lourd et spécifique à votre OS |
| `.gitignore` | **Oui** | Définit ce qui est exclu de git |

---

## 7. Exercice pratique 1

Faites les étapes suivantes dans votre terminal. Après chaque étape, observez ce qui se passe dans le dossier du projet.

**a)** Si ce n'est pas déjà fait, installez `uv` (cf. notebook précédent).

**b)** Créez un nouveau projet nommé `exercice-uv`, en demandant explicitement **Python 3.11** :

```bash
uv init exercice-uv --python 3.11
cd exercice-uv
```

> **Question** : Quels fichiers ont été créés ? Ouvrez `pyproject.toml` et `.python-version` — que contiennent-ils ?

**c)** Exécutez le fichier `main.py` :

```bash
uv run main.py
```

> **Question** : Que s'affiche-t-il ? Quels nouveaux fichiers/dossiers sont apparus dans le projet ?

**d)** Ajoutez la bibliothèque `polars` à votre projet :

```bash
uv add polars
```

> **Question** : Quels fichiers ont été créés ou modifiés ? Ouvrez `pyproject.toml` — qu'est-ce qui a changé ?

**e)** Ajoutez `ruff` à votre projet, mais **en tant que dépendance de développement** :

```bash
uv add --dev ruff
```

> **Question** : Dans quelle section du `pyproject.toml` `ruff` apparaît-il ? Est-ce la même section que `polars` ?

**f)** Vérifiez que tout fonctionne en créant un petit script. Ouvrez `main.py` et remplacez son contenu par :

```python
import polars as pl

df = pl.DataFrame({
    "nom": ["Alice", "Bob", "Charlie"],
    "age": [30, 25, 35],
})

print(df)
```

Puis exécutez-le :

```bash
uv run main.py
```

## 8. Exercice pratique 2

Modifiez le fichier .python-version pour mettre une autre version de python et ré-exécutez le script, que se passe-t-il ?